## Make Sure you Run this Note book with at least Google TPU (For reference I had used A100)

In [15]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
!pip install -q streamlit peft transformers accelerate bitsandbytes

In [17]:
!ls -lh "/content/drive/MyDrive/sft_lora_model_v4"

total 32M
-rw------- 1 root root 1021 May 11 03:32 adapter_config.json
-rw------- 1 root root 1.5K May 11 03:32 chat_template.jinja
-rw------- 1 root root  745 May 11 03:32 tokenizer_config.json
-rw------- 1 root root  32M May 11 03:32 tokenizer.json


In [18]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')
login(token)

In [19]:
%%writefile app.py

import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

st.set_page_config(
    page_title="QLoRA Story Generator",
    page_icon="3:)",
    layout="wide"
)

BASE_MODEL = "google/gemma-3-1b-it"

# IMPORTANT: change this path only if your adapter folder path is different
ADAPTER_DIR = "/content/drive/MyDrive/sft_lora_model_v4"

SFT_MAX_LENGTH = 1024


@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(
        ADAPTER_DIR,
        use_fast=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )

    base_model.resize_token_embeddings(len(tokenizer))

    if getattr(base_model.config, "pad_token_id", None) is None:
        base_model.config.pad_token_id = tokenizer.pad_token_id

    model = PeftModel.from_pretrained(
        base_model,
        ADAPTER_DIR
    )

    model.eval()
    return model, tokenizer


model, tokenizer = load_model()


def build_story_prompt(genre, story_idea):
    return f"""### Instruction:
Write only the story. Do not include titles, markdown, notes, explanations, or extra instructions.

Write a complete, engaging {genre} short story based on the story idea.

Requirements:
- 1 - 2 paragraphs maximum
- Clear beginning, conflict, and resolution
- The final sentence must clearly conclude the story
- No repetition
- No meta-commentary
- No labels like [WP], [RF], [IP], Answer, Response, or Title
- Must enf with a fullstop(.) it shouldn't end with incomplete sentences.

### Story idea:
{story_idea}

### Story:
"""


def generate_story(
    genre,
    story_idea,
    temperature,
    top_p,
    repetition_penalty,
    max_new_tokens
):
    if not story_idea.strip():
        return "Please enter a story idea."

    prompt = build_story_prompt(genre, story_idea)

    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=SFT_MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=True,
            temperature=float(temperature),
            top_p=float(top_p),
            repetition_penalty=float(repetition_penalty),
            no_repeat_ngram_size=5,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    if decoded.startswith(prompt):
        story = decoded[len(prompt):].strip()
    else:
        story = decoded.strip()

    return story


st.title("Genre-Controlled Story Generator using QLoRA")

st.markdown(
    """
    This demo uses a **Gemma 3 1B instruction model fine-tuned with QLoRA**
    to generate short stories conditioned on a selected genre and story idea.
    """
)

with st.sidebar:
    st.header("Generation Settings")

    genre = st.selectbox(
        "Select Genre",
        ["fantasy", "romance", "sci-fi"],
        index=2
    )

    temperature = st.slider("Temperature", 0.1, 1.2, 0.7, 0.1)
    top_p = st.slider("Top-p", 0.1, 1.0, 0.9, 0.05)
    repetition_penalty = st.slider("Repetition Penalty", 1.0, 1.5, 1.15, 0.05)
    max_new_tokens = st.slider("Max New Tokens", 100, 400, 280, 20)

story_idea = st.text_area(
    "Enter a story idea",
    value="Aliens invade the Australian outback, but things do not go according to plan.",
    height=120
)

if st.button("Generate Story", type="primary"):
    with st.spinner("Generating story..."):
        story = generate_story(
            genre=genre,
            story_idea=story_idea,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            max_new_tokens=max_new_tokens
        )

    st.subheader("Generated Story")
    st.write(story)

    with st.expander("Prompt sent to model"):
        st.code(build_story_prompt(genre, story_idea))

st.divider()

st.markdown(
    """
    ### Project Notes

    **Task:** Genre-controlled short story generation
    **Base Model:** Gemma 3 1B instruction model
    **Fine-tuning:** QLoRA
    **Genres:** Fantasy, Romance, Sci-fi
    **Evaluation:** Llama-based judge rubric for genre fidelity, prompt relevance, coherence, ending quality, repetition control, style quality, and format following
    """
)

Overwriting app.py


In [20]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [21]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [22]:
!tail -n 80 streamlit.log



2026-05-11 03:42:45.452 Port 8501 is not available


In [11]:
!./cloudflared tunnel --url http://localhost:8501

2026-05-11T03:40:49Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-11T03:40:49Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-11T03:40:49Z ERR Error unmarshaling QuickTunnel response: error code: 1101 error="invalid character 'e' looking for beginning of value" status_code="500 Internal Server Error"
failed to unmarshal quick Tunnel: invalid character 'e' looking for beginning of value


# Backup if the previous cell failed

In [12]:
!pkill -f cloudflared
!pkill -f localtunnel
!pkill -f ngrok

In [13]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [14]:
!tail -n 50 streamlit.log



2026-05-11 03:41:00.682 Port 8501 is not available


In [23]:
!pip install -q pyngrok

In [24]:
from pyngrok import ngrok

ngrok.set_auth_token("35Rbd6mCiVQ5XJ96YDoyzA2XVcw_2coUuoqHJfxG2K3KMkPFc")

In [25]:
public_url = ngrok.connect(8501)
print("Streamlit app URL:", public_url)

Streamlit app URL: NgrokTunnel: "https://eldora-eponymous-subduedly.ngrok-free.dev" -> "http://localhost:8501"
